In [1]:
#   ---> input (text /mri /labs ) 
                                #---> daignouses
                                #--> search for nearset hospital with the required specilization and your location
                                #--> store cases 

In [ ]:
medical_prompt=""" you are a medical assistant agent that helps patients to find 
the nearst hospital based on their daignoses 
user will upload images or lab results and provide the patient information  .
you have a tool for searching web for nearst hospital based on daignoses and location if user asks for it.
you should use the tools when needed and you should not use them if not needed.
"""

In [ ]:
from tavily import TavilyClient
from langchain.tools import tool
tavily_client = TavilyClient()
@tool
def WebSearch(daignoses:str, location:str)->str:
    """search for the nearst hospital based on daignoses and location"""
    # here you can use any search engine api to get the results
    query = f"nearest hospital for {daignoses} near {location}"
    response = tavily_client.search(query)
    return response

In [15]:
from langchain.agents.middleware import before_agent
from langchain.agents import AgentState
from langgraph.runtime import Runtime 
from langchain.messages import ToolMessage , RemoveMessage
@before_agent
def trim_messages(state:AgentState, runtime:Runtime)->AgentState:
    """ remove all tool messages from the state """
    messages=state['messages']
    trimmed_messages = [msg for msg in messages if not isinstance(msg, ToolMessage)]
    return {'messages': trimmed_messages}


In [16]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
doctor=create_agent(
    model='gpt-4.1',    
    system_prompt=medical_prompt,
    checkpointer=InMemorySaver(),    #--non persistence (testing /debug )
    tools=[WebSearch],
    middleware=[trim_messages])

In [17]:
from langchain.messages import HumanMessage
import base64
with open("imgs/download.jfif", "rb") as f:
    image_base64 = base64.b64encode(f.read()).decode("utf-8")
res=doctor.invoke({
    "messages":[
        HumanMessage(
            content=[
                {
                    "type":"text",
                    "text":"what is the daignoses for this medical image ?"
                },{
                    "type":"image",
                    "base64":image_base64,
                    "mime_type": "image/png"
                }
            ])
            ]},
config={"configurable":{"thread_id":5}
})

In [7]:
res

{'messages': [HumanMessage(content=[{'type': 'text', 'text': 'what is the daignoses for this medical image ?'}, {'type': 'image', 'base64': '/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAkGBxMSEhUSEhIVFhUVFRYVGBUVFRYVFxgWFRUWGBUVFxUYHSggGBolGxcYITEhJSkrLi4uFx8zODMtNygtLisBCgoKDg0OFQ8PFS0dHR0tLSstLS0tLS0tLS0rLS0tLS0tLS0uKy0tLS0rLSsrLS0tLS0tLS0tLS0tLS0tLS0rLf/AABEIAKgBLAMBIgACEQEDEQH/xAAbAAABBQEBAAAAAAAAAAAAAAADAAECBAUGB//EAEIQAAEDAgMEBwUGBAQHAQAAAAEAAhEDIQQSMQVBUWEGEyJxgZGhMlKxwfAUI0Ji0eEzcoLxB5KishUkU2Nzo7MW/8QAFwEBAQEBAAAAAAAAAAAAAAAAAAECA//EAB4RAQEBAAMBAAMBAAAAAAAAAAABEQIhMRIiQVED/9oADAMBAAIRAxEAPwDiQpSoSnCNphOoBPKCYUghgojVBIJJJEqhJJJBAkykmhApTSkVGVBKU0ppTIJSlKZOgda+OpBlNoG9snvIH14rOwNLM9o4laO2ameplGjQAO4IBYPDWnkT689VCh2a0/lqf/N4Wthmw0xxyyOGvyVCowGsI/EHeZaQozrssBhMrGjg1o/0o2Po5XUz+Vp8nFaJpxIGu7w0Udv0cvV/+P4EE/FRlV6mHAgb1S26z7l3NzB5uC1t/j9fXNUNvDsMHGrT/wB11Vcj07f/AM24e6xjfKVzhK2elz5xdXvA9AsRxRqeBvKhKVQqBKKeUxKaUxcqHJTEqJKbMgmTZNmUCVElAQuSzIWZLMgvJwoAqQVE0xKaUggmCiNKEFMICSmlRlJBOUgVGU0qCeZOXocpSgk

In [8]:
print(res['messages'][-1].content)

This is an MRI image of the ankle. Based on the image you’ve provided, it appears to show changes that could be consistent with an Achilles tendon injury, such as a partial or complete tear, or significant inflammation (tendinopathy). There may also be signs of soft tissue swelling and possible abnormalities in the surrounding bone structures, but precise interpretation would require a radiologist's evaluation and consideration of clinical symptoms.

If you have any specific symptoms or need a more detailed diagnosis, please provide more information or consult with your healthcare provider. If you'd like help finding a nearby hospital specializing in this type of injury, let me know your location.


In [11]:
res=doctor.invoke({
    "messages":[
        HumanMessage(
            content="Aswan")
            ]},
config={"configurable":{"thread_id":5}
})

In [12]:
res

{'messages': [HumanMessage(content=[{'type': 'text', 'text': 'what is the daignoses for this medical image ?'}, {'type': 'image', 'base64': '/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAkGBxMSEhUSEhIVFhUVFRYVGBUVFRYVFxgWFRUWGBUVFxUYHSggGBolGxcYITEhJSkrLi4uFx8zODMtNygtLisBCgoKDg0OFQ8PFS0dHR0tLSstLS0tLS0tLS0rLS0tLS0tLS0uKy0tLS0rLSsrLS0tLS0tLS0tLS0tLS0tLS0rLf/AABEIAKgBLAMBIgACEQEDEQH/xAAbAAABBQEBAAAAAAAAAAAAAAADAAECBAUGB//EAEIQAAEDAgMEBwUGBAQHAQAAAAEAAhEDIQQSMQVBUWEGEyJxgZGhMlKxwfAUI0Ji0eEzcoLxB5KishUkU2Nzo7MW/8QAFwEBAQEBAAAAAAAAAAAAAAAAAAECA//EAB4RAQEBAAMBAAMBAAAAAAAAAAABEQIhMRIiQVED/9oADAMBAAIRAxEAPwDiQpSoSnCNphOoBPKCYUghgojVBIJJJEqhJJJBAkykmhApTSkVGVBKU0ppTIJSlKZOgda+OpBlNoG9snvIH14rOwNLM9o4laO2ameplGjQAO4IBYPDWnkT689VCh2a0/lqf/N4Wthmw0xxyyOGvyVCowGsI/EHeZaQozrssBhMrGjg1o/0o2Po5XUz+Vp8nFaJpxIGu7w0Udv0cvV/+P4EE/FRlV6mHAgb1S26z7l3NzB5uC1t/j9fXNUNvDsMHGrT/wB11Vcj07f/AM24e6xjfKVzhK2elz5xdXvA9AsRxRqeBvKhKVQqBKKeUxKaUxcqHJTEqJKbMgmTZNmUCVElAQuSzIWZLMgvJwoAqQVE0xKaUggmCiNKEFMICSmlRlJBOUgVGU0qCeZOXocpSgk

In [13]:
print(res['messages'][-1].content)

For treatment of an Achilles tendon injury in Aswan, you can consult orthopedic specialists at local hospitals. One notable orthopedic consultant in Aswan is Dr. Osama Thapet, who specializes in orthopedic surgery, fractures, and joint surgery.

You may visit the main government or private hospitals in Aswan for urgent care or orthopedic treatment:
- Aswan University Hospital
- Aswan General Hospital

If you need more specific directions or want to book an orthopedic specialist, please let me know!


In [14]:
res

{'messages': [HumanMessage(content=[{'type': 'text', 'text': 'what is the daignoses for this medical image ?'}, {'type': 'image', 'base64': '/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAkGBxMSEhUSEhIVFhUVFRYVGBUVFRYVFxgWFRUWGBUVFxUYHSggGBolGxcYITEhJSkrLi4uFx8zODMtNygtLisBCgoKDg0OFQ8PFS0dHR0tLSstLS0tLS0tLS0rLS0tLS0tLS0uKy0tLS0rLSsrLS0tLS0tLS0tLS0tLS0tLS0rLf/AABEIAKgBLAMBIgACEQEDEQH/xAAbAAABBQEBAAAAAAAAAAAAAAADAAECBAUGB//EAEIQAAEDAgMEBwUGBAQHAQAAAAEAAhEDIQQSMQVBUWEGEyJxgZGhMlKxwfAUI0Ji0eEzcoLxB5KishUkU2Nzo7MW/8QAFwEBAQEBAAAAAAAAAAAAAAAAAAECA//EAB4RAQEBAAMBAAMBAAAAAAAAAAABEQIhMRIiQVED/9oADAMBAAIRAxEAPwDiQpSoSnCNphOoBPKCYUghgojVBIJJJEqhJJJBAkykmhApTSkVGVBKU0ppTIJSlKZOgda+OpBlNoG9snvIH14rOwNLM9o4laO2ameplGjQAO4IBYPDWnkT689VCh2a0/lqf/N4Wthmw0xxyyOGvyVCowGsI/EHeZaQozrssBhMrGjg1o/0o2Po5XUz+Vp8nFaJpxIGu7w0Udv0cvV/+P4EE/FRlV6mHAgb1S26z7l3NzB5uC1t/j9fXNUNvDsMHGrT/wB11Vcj07f/AM24e6xjfKVzhK2elz5xdXvA9AsRxRqeBvKhKVQqBKKeUxKaUxcqHJTEqJKbMgmTZNmUCVElAQuSzIWZLMgvJwoAqQVE0xKaUggmCiNKEFMICSmlRlJBOUgVGU0qCeZOXocpSgk

In [ ]:
#---context / cost optmization / context window 
# --->trim messages 
#----->summerize messages